## CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Fall 2025


# Homework 1: Let's GOOOOO!

- **100 points [7.5% of your final grade]**
- **Due Tuesday, September 14 by 11:59pm**

***Goals of this homework:***
1. Collect data from the web, clean it, and then make some observations based on exploratory data analysis
2. Understand and implement the classic apriori algorithm and extensions to find the association rules in a movie rating dataset
3. Push yourself by using Spark to find association rules

***Submission instructions:***

You should post your notebook to Canvas (look for the homework 1 assignment there). Please name your submission **your-uin_hw1.ipynb**, so for example, my submission would be something like **555001234_hw1.ipynb**. Your notebook should be fully executed when you submit ... so run all the cells for us so we can see the output, then submit that.

***Late Days:***

As a reminder, you begin the semester with five late days. You may use as many as you like. There is no need to alert us to how many late days you are using. Just submit and we will make note of it. Also remember that once your late days are used up, homeworks will receive a 0.

***Collaboration and AI Assistance declaration:***

If you worked with someone on this homework, please be sure to mention that. Remember to include citations to any sources you use in the homework. Also tell us what AI assistant you used and how you used it.

## (REQUIRED) Collaboration and AI Assistance Declaration

### Collaboration Declaration:

*your response goes here*

### AI Assistance Declaration:

*your response goes here*


## (45 points) Part 1: UFO Sightings — Data Ingestion, Cleaning, and Feature Engineering

**Dataset:** `ufos.csv`

Detected columns: `datetime`, `city`, `state`, `country`, `shape`, `duration (seconds)`, `duration (hours/min)`, `comments`, `date posted`, `latitude`, `longitude`, and possibly extra unnamed columns.

**Goal:** Load the data, diagnose issues, clean/standardize it, and derive basic features to support downstream mining. You will probably want to use `pandas` for this.



### (5pts) Part 1a: Load and sanity-check the raw data

Tasks:
1. Load `ufos.csv` into a DataFrame named `ufo_raw`.
2. Display 5 random rows and `ufo_raw.info()`.
3. Briefly report the number of rows/columns and any obviously empty columns.



### (5pts) Part 1b: Clean up datatypes & columns

Create a cleaned DataFrame `ufo`:

- Drop fully-empty or irrelevant columns (e.g., unnamed columns).
- Parse `datetime` to `datetime64[ns]` (`errors='coerce'`).
- Coerce `duration (seconds)`, `latitude`, `longitude` to numeric.
- Lowercase/trim `city`, `state`, `country`, `shape`.
- Remove rows with impossible coordinates (lat ∉ [-90,90], lon ∉ [-180,180]).
- Drop exact duplicates based on a reasonable subset (document your choice).

Provide a short markdown note explaining your choices.


### (5pts) Part 1c: Be thankful!

Now take a look at the `duration (hours/min)` column. In a previous version of this homework, students spent upwards of 10-20 hours just cleaning the values in this column. Luckily for you, the rest of this assignment relies on the much nicer `duration (seconds)` column. For this question, we'd like you to just extract how ever many versions of durations reported in *minutes* you can from the `duration (hours/min)` column. In other words, find as many different variations of anything that could be reasonably interpreted as a minute-like duration. Examples include:

* several minutes
* x to y minutes
* x minutes
* x min.
* x mins.
* and so on ...



### (10pts) Part 1d: Feature engineering and small exploratory data analysis

Create additional columns in a new feature engineering version of the dataset called `ufo_fe`:

- `year`, `month`, `hour` from `datetime`
- `duration_log10` = log10(duration_seconds + 1)
- `duration_bucket` = categorical bins for `duration (seconds)` (you may adjust edges)
- `us_only` = 1 if `country == 'us'` else 0

Then produce:
- Value counts of `shape` (top 10).
- A table of sightings by `year` (counts).
- A `state × shape` table (top 10 states by sample size).



### (5pts) Part 1e: Observations and conclusions

In 3–6 sentences, summarize data quality and distributional patterns; reference at least one artifact from 1d.



### (5pts) Part 1e: Next steps

Propose 2–3 concrete next steps (e.g., better deduplication with fuzzy text, geospatial clustering, normalization of duration text, timezone handling) to improve the quality of the data before moving on to some downstream tasks (you don't need to implement these).



### (10 points) Part 1 f: Scalable & Structured Patterns

Now let's explore **network**, **geospatial**, and **time** patterns to dig a little deeper into our data.
Complete **any two** sub‑tasks below with **pandas**. For each you should provide your code, the ouput, plus some written analysis of what you find.



#### Sub-task 1. Co‑witness graph (space‑time co‑occurrence)
Bucket by geocell (`lat1=round(latitude,1)`, `lon1=round(longitude,1)`) and `datetime` floored to 15 minutes.  
Two reports in the same bucket are co‑witnessed. Build edges between reports in each bucket and summarize the graph (top buckets, approximate largest component).



#### Sub-task 2.  Geospatial hotspots
Aggregate by (`lat1`,`lon1`) and list **top‑20** cells. Optionally justify a normalization (per capita proxies or surface area) if you apply one.



#### Sub-task 3.  Shape × color co‑occurrence
Extract color keywords from `comments` (red/orange/yellow/green/blue/purple/violet/white/black/silver/gold/pink; handle “-ish” variants). Build a co‑occurrence table with `shape` or your normalized `shape_norm`. Interpret one pairing.



#### Sub-task 4. Spatio‑temporal spikes (anomaly cues)
For each geocell, create a daily count series; compute z‑scores within cell; list top‑10 spikes across all cells. Discuss plausible causes.


## (40 points) Part 2: Association Rules in Movie Rating Behaviors

For the second part of this homework, we're going to examine movies using our understanding of association rules, to find movies that "go together". For this part, you will implement the apriori algorithm, and apply it to a movie rating dataset. We'll use the [MovieLens](https://grouplens.org/datasets/movielens/) dataset.

First, run the next cell to load the dataset we are going to use.

In [ ]:
import urllib3
import zipfile

http = urllib3.PoolManager()
req = http.request("GET", "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", preload_content=False)

with open("movie.zip", 'wb') as out:
  while True:
    data = req.read(4096)
    if not data:
      break
    out.write(data)
req.release_conn()

zFile = zipfile.ZipFile("movie.zip", "r")
for fileM in zFile.namelist():
  zFile.extract(fileM)

In [ ]:
!ls ml-latest-small/

In this dataset, there are four columns: `userId` is the integer ids of users, `movieId` is the integer ids of movies, `rating` is the rate of the user gives to the movie, and `timestamp` which we do not use here. Each row denotes that the user of given `userId` rated the movie of the given `movieId`. We are going to treat each user as a "basket", so you will need to collect all the movies that have been rated by a single user as a basket.

Now, you need to implement the apriori algorithm and apply it to this dataset to find association rules of user rating behaviors where:

1. Define `rating` >= 3 is "like" (that is, only consider movie ratings of 3 or higher in your baskets; you may ignore all others)
2. `minsup` == 40 (out of 600 users/baskets); we may adjust this based on the discussion on Canvas
3. `minconf` == to be determined by a discussion on Canvas. You may try several different choices, but we will converge on a good choice for everyone for the final submission.

We know there are many existing implementations of apriori online (check github for some good starting points). You are welcome to read existing codebases and let that inform your approach. Do not copy-paste any existing code. We want your code to have sufficient comments to explain your steps, to show us that you really know what you are doing. Furthermore, you should add print statements to print out the intermediate steps of your method -- e.g., the size of the candidate set at each step of the method, the size of the filtered set, and any other important information you think will highlight the method.

To help get you started, we can load the ratings with the following code snippet:

In [ ]:
import pandas as pd
# read user ratings
allRatings = pd.read_csv("ml-latest-small/ratings.csv")
allRatings

### (15pts) Step 1: Implement Apriori Algorithm
In this section, you need to implement the Apriori algorithm, we will check the correctness of your code and we encourage efficient implementation.

In [ ]:
# your code here

### (5pts) Step 2: Print Your Association Rules

Next you should print your final association rules in the following format:

**movie_name_1, movie_name_2, ... -->
movie_name_k**

where the movie names can be fetched by joining the movieId with the file `movies.csv`. For example, one rule that you might find is:

**Matrix, The (1999),  Star Wars: Episode V - The Empire Strikes Back (1980),  Star Wars: Episode IV - A New Hope (1977),  ->
Star Wars: Episode VI - Return of the Jedi (1983)**

In [ ]:
# your code here

### (10pts) Step 3: Implement Random Sampling

We discussed in class a method to randomly sample baskets to avoid the overhead of reading the entire set of baskets (which in practice, could amount to billions of baskets). For this part, you should implement such a random sampling approach that takes a special parameter **alpha** that controls the size of the sample: e.g., alpha = 0.10 means to sample 10% of the baskets (our users, in this case).

Vary **alpha** and report the number of frequent itemsets you find and how this compares to the number of frequent itemsets in the entire dataset. What do you discover?


In [ ]:
# your code here

*your discussion here*

### (10pts) Step 4: Check for False Positives

Next you should verify that the candidate pairs you discover by random sampling are truly frequent by comparing to the itemsets you discover over the entire dataset.

For this part, consider another parameter **minsup_sample** that relaxes the minimum support threshold. For example if we want minsup = 1/100 for whole dataset, then try minsup_sample = 1/125 for the sample. This will help catch truly frequent itemsets.

Vary **minsup_sample** and report the number of frequent itemsets you find and the number of false positives you find. What do you discover?


In [ ]:
# your code here

*your discussion here*

## (5 points) Part 3: Spark-based Association Rules

So far, we have been working with a fairly small dataset. For this last question, you should use the much larger **Movies 10M** dataset: https://files.grouplens.org/datasets/movielens/ml-10m.zip

First, we need to load this larger dataset:

In [ ]:
import urllib3
import zipfile

http = urllib3.PoolManager()
req = http.request("GET", "https://files.grouplens.org/datasets/movielens/ml-10m.zip", preload_content=False)

with open("movie.zip", 'wb') as out:
  while True:
    data = req.read(4096)
    if not data:
      break
    out.write(data)
req.release_conn()

zFile = zipfile.ZipFile("movie.zip", "r")
for fileM in zFile.namelist():
  zFile.extract(fileM)

Now, see if you can write a Spark-based implementaiton of Apriori. You'll see that the Spark library MLib does contain a "Frequent Pattern Mining" implementation. You may not use this! Good luck!

In [ ]:
# your code here